In [1]:
import yaml
import logging
import random
import pandas as pd
import mlflow
import datasets
import torch
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
from shared import utils

# logging.basicConfig(filename='sentiment_analysis.log', level=logging.INFO)
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# GLOBAL VARIABLES
config_file = "../configs/run1_sentiment.yaml"
with open(config_file, "r") as f:
    config = yaml.load(f, Loader=yaml.SafeLoader)
dataset_config = config.get("dataset")
model_config = config.get("model")
device = 'cuda' if torch.cuda.is_available() else 'cpu'

rng = random.Random(config.get('seed', 0))

logger.info(f"Configs loaded. Device is set to {device}")

In [3]:
experiment_name = config.get("mlflow", dict).get("experiment", None)
if experiment_name is None:
    raise ValueError(
        "Experiment name is not set in the config.\n"
        "Notebook does not run without tracking"
    )
mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='mlflow-artifacts:/13', creation_time=1791120060086, effective_trace_archival_retention=None, experiment_id='13', last_update_time=1791120060086, lifecycle_stage='active', name='llm-bias-variance-2', tags={}, trace_location=None, workspace='default'>

In [4]:
# prep data
dataset=datasets.load_dataset(dataset_config["name"])
split = dataset["train"].train_test_split(train_size=0.7)
train_dataset = split['train']
val_dataset = split['test']
test_dataset = dataset['test']

# Model and Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_config["name"], padding_side='left')
model = AutoModelForCausalLM.from_pretrained(model_config["name"]).to(device)
model.eval()

# Generation Config
generation_config = GenerationConfig(**config.get("generation"))

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 3586.25it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [6]:
induction_prompt = """You are given a list of inputs and outputs. Your job is to 
infer the nature of the task from the given samples and create instructions for an LLM to correctly complete this task."""
k = 5
selected_samples = rng.choices(train_dataset, k=k)
with mlflow.start_run():
    samples_with_layout = "\n\n".join([f"Input: {sample['review']}\nOutput: {sample['label']}" for sample in selected_samples])
    full_prompt = f"## Instruction\n\n{induction_prompt}\n\n## Samples\n\n{samples_with_layout}\n\n### Induction Prompt\n\n" 
    full_prompt_chat_template = [
        {
            "role": "system",
            "content": f"## Instruction\n\n{induction_prompt}\n\nPlace your output under the headline ### Induction Prompt"
        },
        {
            "role": "user",
            "content": f"\n\n## Samples\n\n{samples_with_layout}"
        }
    ]
    
    # tokenize and generate
    if config.get("chat_template", False):
        tokens = tokenizer.apply_chat_template(full_prompt_chat_template, return_tensors="pt") 
    else:
        tokens = tokenizer([full_prompt], return_tensors="pt")
    tokens = tokens.to(model.device)
    with torch.no_grad():
        generated_ids = model.generate(**tokens, generation_config=generation_config)
    output = tokenizer.decode(generated_ids, skip_special_tokens=True)
    
    # Induced Prompt
    generation_chunks = output[0].split("### Induction Prompt")
    if len(generation_chunks) == 1:
        ValueError("No separate induction prompt can be extracted")
    induced_prompt = generation_chunks[-1]

    # log params
    mlflow.log_params(dict(num_samples=k, **model_config, **config.get("generation", {})))

    # log prompts
    mlflow.log_text(induction_prompt, "induction_prompt.txt")
    mlflow.log_text(output, "completed_induction_prompt.txt")
    mlflow.log_text(induced_prompt, "induced_prompt.txt")
    
    # Validation
    val_batch_dataset = val_dataset.batch(
        batch_size=config.get("validation", {}).get("batch_size", 32),
        drop_last_batch=False,
    )
    with torch.no_grad():
        inputs = []
        ground_truth = []
        predictions = []
        for batch in val_batch_dataset:
            # Prepare inputs
            reviews = [
                [
                    {
                        "role": "system",
                        "content": f"{induced_prompt}\n Place your output under the headline ### Output"
                    },
                    {
                        "role": "user",
                        "content": sample
                    }
                ]
                for sample in batch['review']
            ]
            tokens = tokenizer.apply_chat_template(
                reviews, 
                return_tensors="pt",
                padding=True
            ).to(model.device)
    
            # extract outputs
            generated_ids = model.generate(**tokens, generation_config=generation_config)
            full_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
            outputs = [
                utils.coerce_numeric(
                    x.split("### Output")[-1].strip()
                )
                for x in full_text
            ]
            ground_truth.extend(batch['label'])
            inputs.extend(batch['review'])
            break
    accuracy = (np.array(ground_truth) == np.array(outputs)).mean()
    mlflow.log_metric("accuracy", accuracy)